## 1.0 Environment Setup & Data Loading

In [37]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

In [38]:
data_path = "online_shoppers_intention.csv"

if not os.path.exists(data_path):
    url = "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip"
    zip_path = "dataset.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Dataset downloaded and extracted.")

df = pd.read_csv(data_path)
print(f"Raw Dataset Shape: {df.shape}")

# Delete the zip file
file_path = "dataset.zip"

# Check if the file exists before trying to delete it
if os.path.exists(file_path):
    os.remove(file_path)

Raw Dataset Shape: (12330, 18)


## 2.0 Feature Engineering

In [39]:
# Compute intermediate totals
total_pages = df['Administrative'] + df['Informational'] + df['ProductRelated']
total_duration = (
    df['Administrative_Duration'] 
    + df['Informational_Duration'] 
    + df['ProductRelated_Duration']
)

# 1. Average dwell time per page
df['Avg_Duration_Per_Page'] = total_duration / (total_pages + 1e-5)

# 2. Product browsing focus ratio
df['Product_Ratio'] = df['ProductRelated'] / (total_pages + 1e-5)

# Separate features and target
X = df.drop(columns=['Revenue'])
y = df['Revenue'].astype(int)

## 3.0 Train/Test Split

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=45, stratify=y
)

# Export test partition with target for Streamlit upload
test_df = X_test.copy()
test_df['Revenue'] = y_test
test_df.to_csv("test_data.csv", index=False)
print("Engineered test_data.csv saved successfully with shape:", test_df.shape)

Engineered test_data.csv saved successfully with shape: (2466, 20)


## 4.0 Preprocessing Pipeline & Model Definitions

In [41]:
# Categorical columns: standard categorical + integer-coded IDs
categorical_cols = [
    'Month', 'VisitorType', 'Weekend',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]

# Continuous numerical columns including the 2 engineered features
numerical_cols = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
    'Avg_Duration_Per_Page', 'Product_Ratio'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', PowerTransformer(method='yeo-johnson'), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols)
    ]
)

## 5.0 Model Training, Metric Evaluation, and Export

In [42]:
models = {
    "Logistic Regression": Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=45))
    ]),
    "Decision Tree": Pipeline([
        ('prep', preprocessor),
        ('clf', DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=20, random_state=45))
    ]),
    "kNN": Pipeline([
        ('prep', preprocessor),
        ('clf', KNeighborsClassifier(n_neighbors=9, weights='distance', metric='manhattan'))
    ]),
    "Naive Bayes": Pipeline([
        ('prep', preprocessor),
        ('clf', GaussianNB())
    ]),
    "Support Vector Machine": Pipeline([
        ('prep', preprocessor),
        ('clf', SVC(C=0.8, kernel='rbf', class_weight='balanced', probability=True, random_state=45))
    ])
}

### 6.0 Train, Evaluate, and Save Artifacts

In [43]:
os.makedirs("model", exist_ok=True)
metrics_records = []

for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    
    # Save model pipeline
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(pipeline, f"model/{safe_name}.joblib")
    
    # Predict and evaluate
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    metrics_records.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

# Format comparison table
metrics_df = pd.DataFrame(metrics_records)
print("\nEvaluation Comparison Table:")
metrics_df

Training Logistic Regression...
Training Decision Tree...


/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/

Training kNN...


/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Training Naive Bayes...
Training Support Vector Machine...


/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-onl


Evaluation Comparison Table:


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8654,0.9236,0.5421,0.8429,0.6598,0.6028
1,Decision Tree,0.8682,0.9280,0.5501,0.8194,0.6583,0.5982
2,kNN,0.9039,0.9095,0.7458,0.5759,0.6499,0.6020
3,Naive Bayes,0.2178,0.7677,0.1633,0.9817,0.2800,0.0851
4,Support Vector Machine,0.8670,0.9295,0.5453,0.8508,0.6646,0.6091


In [44]:
import sklearn
print(sklearn.__version__)

1.9.1
